In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
import pandas as pd
import plotly.express as px
import plotly.graph_objs as go

# Load your CSV file
file_path = '/content/potoro.csv'
historical_data = pd.read_csv(file_path)

# Convert timestamp string to datetime object
historical_data['Timestamp'] = pd.to_datetime(historical_data['Timestamp'])

# Extract time component in seconds since midnight
historical_data['Time'] = historical_data['Timestamp'].dt.hour * 3600 + historical_data['Timestamp'].dt.minute * 60 + historical_data['Timestamp'].dt.second

# Example of a regular new transaction
new_data = {
    'Amount': 100.00,
    'Timestamp': '2024-04-18 02:00:00',  # Example timestamp
    'Domain': 'Food'  # Example domain
}

# Simulating a fraudulent transaction
fraudulent_data = {
    'Amount': 10000.00,  # Unusually large amount (can change to simulate fraud)
    'Timestamp': '2024-04-18 03:00:00',  # Different timestamp
    'Domain': 'Food'  # Same domain for comparison
}

# Filter historical data by domain (for regular new transaction)
domain_data = historical_data[historical_data['Domain'] == new_data['Domain']].copy()

# Drop rows with missing values
domain_data.dropna(inplace=True)

# Standardize the features
scaler = StandardScaler()
X = domain_data[['Amount', 'Time']]
X_scaled = scaler.fit_transform(X)

# Train the Isolation Forest model
model = IsolationForest(contamination=0.01)
model.fit(X_scaled)

# Convert timestamp string to datetime object for regular and fraudulent transactions
new_data['Timestamp'] = pd.to_datetime(new_data['Timestamp'])
fraudulent_data['Timestamp'] = pd.to_datetime(fraudulent_data['Timestamp'])

# Extract time component for both new and fraudulent data
new_data['Time'] = new_data['Timestamp'].hour * 3600 + new_data['Timestamp'].minute * 60 + new_data['Timestamp'].second
fraudulent_data['Time'] = fraudulent_data['Timestamp'].hour * 3600 + fraudulent_data['Timestamp'].minute * 60 + fraudulent_data['Timestamp'].second

# Scale the features of the new and fraudulent data
new_data_scaled = scaler.transform([[new_data['Amount'], new_data['Time']]])
fraudulent_data_scaled = scaler.transform([[fraudulent_data['Amount'], fraudulent_data['Time']]])

# Predict if the new data and fraudulent data are anomalies
is_new_anomaly = model.predict(new_data_scaled)
is_fraudulent_anomaly = model.predict(fraudulent_data_scaled)

# Create a DataFrame for the new and fraudulent transactions
transactions = pd.DataFrame({
    'Amount': [new_data['Amount'], fraudulent_data['Amount']],
    'Time': [new_data['Time'], fraudulent_data['Time']],
    'Type': ['New Transaction', 'Fraudulent Transaction'],
    'Is_Anomaly': [is_new_anomaly[0], is_fraudulent_anomaly[0]]
})

# Print DataFrame after filtration
print("\nDomain Data After Filtration:")
print(domain_data)

# Mark anomalies and normal data for plotting
domain_data['Type'] = 'Historical Data'
domain_data['Is_Anomaly'] = 1  # Historical data assumed to be normal

# Combine historical and new/fraudulent data
plot_data = pd.concat([domain_data[['Amount', 'Time', 'Type', 'Is_Anomaly']], transactions])

# Plot using Plotly
fig = px.scatter(
    plot_data,
    x='Time',
    y='Amount',
    color='Type',
    symbol='Is_Anomaly',
    hover_data=['Amount', 'Time'],
    title='Amount vs Time with Anomalies',
    labels={'Time': 'Time (Seconds)', 'Amount': 'Transaction Amount'},
    color_discrete_map={
        'Historical Data': 'blue',
        'New Transaction': 'red',
        'Fraudulent Transaction': 'green'
    },
    symbol_map={1: 'circle', -1: 'x'}  # -1 is an anomaly, 1 is normal
)

# Update layout for better visibility
fig.update_layout(
    xaxis_title="Time (Seconds since midnight)",
    yaxis_title="Amount",
    legend_title="Transaction Type",
    template="plotly_white",
    hovermode="closest"
)

# Show the interactive plot
fig.show()

# Output results for both transactions
if is_new_anomaly == -1:
    print("The regular new transaction is an anomaly.")
else:
    print("The regular new transaction is not an anomaly.")

if is_fraudulent_anomaly == -1:
    print("The fraudulent transaction is an anomaly.")
else:
    print("The fraudulent transaction is not an anomaly.")
